# AI vs Human Text Detection
## Project 1 – Machine Learning & Deep Learning Classification
**Course:** Introduction to AI & Agents | **Summer I 2026**

This notebook covers all four required sections:
1. Data Exploration & Preprocessing
2. Feature Engineering (TF-IDF, Word2Vec, GloVe)
3. Model Training & Tuning (3 ML + 3 DL models)
4. Evaluation & Comparison


---
# Section 1 – Data Exploration & Preprocessing


## 1.1 Import Libraries


In [ ]:
# Install all required packages
!pip install gensim wordcloud nltk scikit-learn pandas numpy matplotlib seaborn joblib tensorflow torch torchvision torchaudio openpyxl -q


In [ ]:
import os, re, pickle, json, warnings
warnings.filterwarnings('ignore')
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import matplotlib
matplotlib.rcParams['figure.figsize'] = (10, 6)

import nltk
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, roc_auc_score, roc_curve,
    classification_report, ConfusionMatrixDisplay
)
import joblib

from gensim.models import Word2Vec
import gensim.downloader as gensim_api

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Download NLTK resources
# Download ALL required NLTK resources explicitly
nltk_resources = [
    "stopwords", "punkt", "punkt_tab", "wordnet",
    "omw-1.4", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"
]
for r in nltk_resources:
    nltk.download(r, quiet=True)
print("NLTK resources downloaded.")

print("All libraries loaded successfully!")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## 1.2 Configuration & Paths


In [ ]:
# ─── SET YOUR DATA PATH HERE ──────────────────────────────────────────────
DATA_FILE = r'C:\Users\hamza\Downloads\train_data with labels.xlsx'
# ──────────────────────────────────────────────────────────────────────────

os.makedirs('models', exist_ok=True)
os.makedirs('plots', exist_ok=True)
os.makedirs('reports', exist_ok=True)
print('Directories ready: models/, plots/, reports/')


## 1.3 Load Dataset


In [ ]:
df = pd.read_excel(DATA_FILE)

# Rename to project convention (text->essay, label stays)
df.columns = ['essay', 'label']

print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'\nLabel distribution (0=Human, 1=AI):')
print(df['label'].value_counts())
print(f'\nMissing values: {df.isnull().sum().sum()}')
df.head(3)


## 1.4 Exploratory Data Analysis

### 1.4.1 Class Distribution


In [ ]:
counts = df['label'].value_counts().sort_index()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(['Human (0)', 'AI (1)'], counts.values,
            color=['steelblue', 'salmon'], edgecolor='black')
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v+30, str(v), ha='center', fontsize=12)

axes[1].pie(counts.values, labels=['Human Written', 'AI Written'],
            autopct='%1.1f%%', colors=['steelblue', 'salmon'], startangle=90)
axes[1].set_title('Class Balance', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('plots/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dataset is perfectly balanced — accuracy is a fair metric here.')


### 1.4.2 Text Length Analysis


In [ ]:
df['text_length'] = df['essay'].str.split().str.len()
df['char_length'] = df['essay'].str.len()

print('=== Word Count Statistics by Label ===')
print(df.groupby('label')['text_length'].describe().round(1))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['text_length'], bins=40, color='mediumseagreen', edgecolor='black', alpha=0.8)
axes[0].set_title('Word Count Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Words'); axes[0].set_ylabel('Frequency')

for label, color, name in [(0,'steelblue','Human'),(1,'salmon','AI')]:
    axes[1].hist(df[df['label']==label]['text_length'], bins=40,
                 alpha=0.6, color=color, label=name, edgecolor='black')
axes[1].set_title('Word Count by Label', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Words'); axes[1].legend()

plt.tight_layout()
plt.savefig('plots/length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


## 1.5 Text Preprocessing

Three steps applied to every document:
1. **Lowercase** + remove punctuation and numeric tokens
2. **Stopword removal** + filter non-alpha / single-char tokens
3. **POS-aware lemmatization** using WordNet


In [ ]:
stop_words = set(stopwords.words('english'))

def text_preprocessing(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\b[\d\w]*\d[\d\w]*\b', '', text)
    text = text.strip()
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words and w.isalpha() and len(w) > 1]
    return ' '.join(tokens)

def get_wordnet_pos(tag):
    if tag.startswith('J'): return wordnet.ADJ
    elif tag.startswith('V'): return wordnet.VERB
    elif tag.startswith('N'): return wordnet.NOUN
    elif tag.startswith('R'): return wordnet.ADV
    return wordnet.NOUN

def lemmatize_text(text):
    lemmatizer = WordNetLemmatizer()
    tokens = word_tokenize(text)
    pos_tags = nltk.pos_tag(tokens)
    return ' '.join(lemmatizer.lemmatize(w, get_wordnet_pos(t)) for w, t in pos_tags)

print('Preprocessing functions defined.')


In [ ]:
print('Applying preprocessing (1-2 minutes)...')
df['cleaned']    = df['essay'].apply(text_preprocessing)
df['lemmatized'] = df['cleaned'].apply(lemmatize_text)
print('Done!')
df[['essay','cleaned','lemmatized']].head(3)


In [ ]:
vocab_raw   = set(' '.join(df['essay']).split())
vocab_clean = set(' '.join(df['cleaned']).split())
vocab_lem   = set(' '.join(df['lemmatized']).split())

print(f'Raw vocabulary size       : {len(vocab_raw):>7,}')
print(f'After preprocessing       : {len(vocab_clean):>7,}  ({len(vocab_raw)-len(vocab_clean):,} words removed)')
print(f'After lemmatization       : {len(vocab_lem):>7,}  ({len(vocab_clean)-len(vocab_lem):,} further reduced)')


## 1.6 WordCloud & Top Words


In [ ]:
all_text = ' '.join(df['lemmatized'].astype(str))
wc = WordCloud(width=1000, height=450, background_color='white',
               colormap='viridis', max_words=150).generate(all_text)

plt.figure(figsize=(12, 5))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('WordCloud of Lemmatized Corpus', fontsize=16, fontweight='bold')
plt.savefig('plots/WordCloud.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
word_counts = Counter(all_text.split())
top_words = pd.DataFrame(word_counts.most_common(20), columns=['Word','Count'])

plt.figure(figsize=(10, 6))
sns.barplot(data=top_words, y='Word', x='Count', palette='Blues_r')
plt.title('Top 20 Most Frequent Words', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/top_words.png', dpi=150, bbox_inches='tight')
plt.show()


---
# Section 2 – Feature Engineering


## 2.1 Train / Validation / Test Split

Stratified 64 / 16 / 20 split to preserve class balance across all sets.


In [ ]:
X = df[['essay','cleaned','lemmatized']]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.20, random_state=42, stratify=y_train)

print(f'Train      : {len(X_train):,} samples | {y_train.value_counts().to_dict()}')
print(f'Validation : {len(X_val):,} samples | {y_val.value_counts().to_dict()}')
print(f'Test       : {len(X_test):,} samples | {y_test.value_counts().to_dict()}')


## 2.2 TF-IDF Features

TF-IDF converts text to sparse vectors. Unigrams + bigrams are used to capture both
single-word patterns and common two-word phrases that differ between AI and human text.


In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50000,
    min_df=2,
    sublinear_tf=True
)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train['lemmatized'])
X_val_tfidf   = tfidf_vectorizer.transform(X_val['lemmatized'])
X_test_tfidf  = tfidf_vectorizer.transform(X_test['lemmatized'])

print(f'TF-IDF train shape: {X_train_tfidf.shape}')
print(f'Vocabulary size   : {len(tfidf_vectorizer.vocabulary_):,}')


## 2.3 Word2Vec Embeddings

Train a Word2Vec skip-gram model on the training corpus.
Each word gets a 100-dimensional dense vector; semantically similar words cluster together.


In [ ]:
W2V_DIM = 100
train_sentences_w2v = [text.split() for text in X_train['lemmatized']]

print('Training Word2Vec (skip-gram, 100d)...')
w2v_model = Word2Vec(
    sentences=train_sentences_w2v,
    vector_size=W2V_DIM,
    window=5,
    min_count=1,
    workers=4,
    epochs=20,
    sg=1
)
print(f'Word2Vec vocab: {len(w2v_model.wv):,} words, {W2V_DIM}d vectors')
# Sanity check
similar = w2v_model.wv.most_similar('write', topn=5)
print(f"Words most similar to 'write': {similar}")


## 2.4 GloVe Embeddings

Load pre-trained GloVe-100 vectors via gensim's downloader.
> **Note:** First run downloads ~376 MB — requires internet connection.
> Subsequent runs use the local cache (~30 seconds).


In [ ]:
GLOVE_DIM = 100
print('Loading GloVe vectors (glove-wiki-gigaword-100)...')
print('This may download ~376 MB on first run.')
glove_model = gensim_api.load('glove-wiki-gigaword-100')
print(f'GloVe loaded: {len(glove_model):,} words, {GLOVE_DIM}d vectors')


## 2.5 Keras Tokenizer & Padding (for Deep Learning models)


In [ ]:
tokenizer = Tokenizer(filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n')
tokenizer.fit_on_texts(X_train['lemmatized'])
word_index = tokenizer.word_index
VOCAB_SIZE = len(word_index) + 1
print(f'Tokenizer vocabulary: {VOCAB_SIZE:,} tokens')


In [ ]:
X_train_seq = tokenizer.texts_to_sequences(X_train['lemmatized'])
X_val_seq   = tokenizer.texts_to_sequences(X_val['lemmatized'])
X_test_seq  = tokenizer.texts_to_sequences(X_test['lemmatized'])

seq_lengths = [len(s) for s in X_train_seq]
MAX_LEN = int(np.percentile(seq_lengths, 95))
print(f'Sequence lengths  -> Mean: {np.mean(seq_lengths):.1f}, '
      f'Median: {np.median(seq_lengths):.1f}, 95th pct: {MAX_LEN}')
print(f'Using MAX_LEN = {MAX_LEN}')
print(f'NOTE: Update max_len_pad in app.py make_prediction() to {MAX_LEN} if needed')


In [ ]:
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post')
X_val_pad   = pad_sequences(X_val_seq,   maxlen=MAX_LEN, padding='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN, padding='post')

print(f'Padded shapes -> Train: {X_train_pad.shape}, Val: {X_val_pad.shape}, Test: {X_test_pad.shape}')


## 2.6 Build Embedding Matrices

Map the tokenizer's vocabulary to pre-trained vectors. OOV words get small random vectors.


In [ ]:
def build_embedding_matrix(word_index, model, dim):
    matrix = np.zeros((len(word_index) + 1, dim), dtype=np.float32)
    hits, misses = 0, 0
    for word, idx in word_index.items():
        try:
            matrix[idx] = model[word]
            hits += 1
        except KeyError:
            matrix[idx] = np.random.normal(0, 0.1, dim)
            misses += 1
    print(f'  Hits: {hits:,} | Misses (random init): {misses:,}')
    return matrix

print('Building Word2Vec embedding matrix...')
embedding_matrix_w2v = build_embedding_matrix(word_index, w2v_model.wv, W2V_DIM)

print('Building GloVe embedding matrix...')
embedding_matrix_glove = build_embedding_matrix(word_index, glove_model, GLOVE_DIM)

print(f'W2V matrix shape  : {embedding_matrix_w2v.shape}')
print(f'GloVe matrix shape: {embedding_matrix_glove.shape}')


In [ ]:
X_train_tensor = torch.LongTensor(X_train_pad)
X_val_tensor   = torch.LongTensor(X_val_pad)
X_test_tensor  = torch.LongTensor(X_test_pad)
y_train_tensor = torch.FloatTensor(y_train.values)
y_val_tensor   = torch.FloatTensor(y_val.values)
y_test_tensor  = torch.FloatTensor(y_test.values)

BATCH_SIZE = 32
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor),
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val_tensor, y_val_tensor),
                          batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test_tensor, y_test_tensor),
                          batch_size=BATCH_SIZE, shuffle=False)
print(f'DataLoaders ready. Batches -> Train: {len(train_loader)}, '
      f'Val: {len(val_loader)}, Test: {len(test_loader)}')


---
# Section 3 – Model Training & Tuning


## 3.1 Support Vector Machine (SVM)

SVM finds the maximum-margin hyperplane separating classes.
We use a **linear kernel** (ideal for high-dimensional TF-IDF) and tune C via GridSearch.


In [ ]:
pipeline_svm = Pipeline([
    ('vectorizer', TfidfVectorizer(ngram_range=(1,2), max_features=50000,
                                    min_df=2, sublinear_tf=True)),
    ('model', SVC(kernel='linear', probability=True, random_state=42))
])

param_grid_svm = {'model__C': [0.1, 1.0, 10.0]}
gs_svm = GridSearchCV(pipeline_svm, param_grid_svm, cv=3,
                       scoring='f1', n_jobs=-1, verbose=1)

print('GridSearchCV for SVM (may take 3-5 min)...')
gs_svm.fit(X_train['lemmatized'], y_train)
print(f'Best C  : {gs_svm.best_params_}')
print(f'Best F1 : {gs_svm.best_score_:.4f}')
pipeline_svm = gs_svm.best_estimator_


In [ ]:
y_pred_svm_val  = pipeline_svm.predict(X_val['lemmatized'])
y_score_svm_val = pipeline_svm.predict_proba(X_val['lemmatized'])[:,1]
print('SVM Validation Report:')
print(classification_report(y_val, y_pred_svm_val, target_names=['Human','AI']))
print(f'ROC-AUC: {roc_auc_score(y_val, y_score_svm_val):.4f}')


## 3.2 Decision Tree

Decision Trees recursively split data by Gini impurity. Fully interpretable.
We tune max_depth and min_samples_split to control overfitting.


In [ ]:
pipeline_dt = Pipeline([
    ('vectorizer', TfidfVectorizer(ngram_range=(1,2), max_features=50000,
                                    min_df=2, sublinear_tf=True)),
    ('model', DecisionTreeClassifier(random_state=42))
])

param_grid_dt = {
    'model__max_depth'        : [10, 20, None],
    'model__min_samples_split': [2, 5, 10],
}
gs_dt = GridSearchCV(pipeline_dt, param_grid_dt, cv=3,
                      scoring='f1', n_jobs=-1, verbose=1)
print('GridSearchCV for Decision Tree...')
gs_dt.fit(X_train['lemmatized'], y_train)
print(f'Best params : {gs_dt.best_params_}')
print(f'Best F1     : {gs_dt.best_score_:.4f}')
pipeline_dt = gs_dt.best_estimator_


In [ ]:
y_pred_dt_val  = pipeline_dt.predict(X_val['lemmatized'])
y_score_dt_val = pipeline_dt.predict_proba(X_val['lemmatized'])[:,1]
print('Decision Tree Validation Report:')
print(classification_report(y_val, y_pred_dt_val, target_names=['Human','AI']))
print(f'ROC-AUC: {roc_auc_score(y_val, y_score_dt_val):.4f}')


## 3.3 AdaBoost

AdaBoost iteratively combines weak classifiers (shallow trees),
up-weighting misclassified samples each round. We tune n_estimators and learning_rate.


In [ ]:
pipeline_ada = Pipeline([
    ('vectorizer', TfidfVectorizer(ngram_range=(1,2), max_features=50000,
                                    min_df=2, sublinear_tf=True)),
    ('model', AdaBoostClassifier(random_state=42))
])

param_grid_ada = {
    'model__n_estimators' : [50, 100, 200],
    'model__learning_rate': [0.5, 1.0],
}
gs_ada = GridSearchCV(pipeline_ada, param_grid_ada, cv=3,
                       scoring='f1', n_jobs=-1, verbose=1)
print('GridSearchCV for AdaBoost...')
gs_ada.fit(X_train['lemmatized'], y_train)
print(f'Best params : {gs_ada.best_params_}')
print(f'Best F1     : {gs_ada.best_score_:.4f}')
pipeline_ada = gs_ada.best_estimator_


In [ ]:
y_pred_ada_val  = pipeline_ada.predict(X_val['lemmatized'])
y_score_ada_val = pipeline_ada.predict_proba(X_val['lemmatized'])[:,1]
print('AdaBoost Validation Report:')
print(classification_report(y_val, y_pred_ada_val, target_names=['Human','AI']))
print(f'ROC-AUC: {roc_auc_score(y_val, y_score_ada_val):.4f}')


## 3.4 Deep Learning Model Architectures

Three PyTorch models — all use a sigmoid output for binary classification.
> **Important:** These class definitions must match `models_dl.py` so `app.py` can load the weights.


In [ ]:
class RNNTextClassifierV2(nn.Module):
    """Bidirectional GRU – captures context in both directions."""
    def __init__(self, embedding_matrix, hidden_dim=128, fc_hidden_dim=64, num_layers=1):
        super().__init__()
        vocab_size, embedding_dim = embedding_matrix.shape
        self.embedding = nn.Embedding.from_pretrained(
            torch.FloatTensor(embedding_matrix), freeze=False)
        self.rnn = nn.GRU(embedding_dim, hidden_dim, num_layers,
                          batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.5)
        self.fc1 = nn.Linear(hidden_dim * 2, fc_hidden_dim)
        self.fc2 = nn.Linear(fc_hidden_dim, 1)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        x = torch.mean(out, dim=1)  # global average pooling
        x = self.dropout(x)
        x = torch.relu(self.fc1(x))
        return torch.sigmoid(self.fc2(x))


class LSTMTextClassifier(nn.Module):
    """Unidirectional LSTM – captures long-range sequential dependencies."""
    def __init__(self, embedding_matrix, hidden_dim=128, fc_hidden_dim=64):
        super().__init__()
        vocab_size, embedding_dim = embedding_matrix.shape
        self.embedding = nn.Embedding.from_pretrained(
            torch.FloatTensor(embedding_matrix), freeze=False)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim,
                            num_layers=1, batch_first=True, bidirectional=False)
        self.dropout = nn.Dropout(0.5)
        self.fc1 = nn.Linear(hidden_dim, fc_hidden_dim)
        self.fc2 = nn.Linear(fc_hidden_dim, 1)

    def forward(self, x):
        x = self.embedding(x)
        _, (hn, _) = self.lstm(x)
        x = self.dropout(hn[-1])
        x = torch.relu(self.fc1(x))
        return torch.sigmoid(self.fc2(x))


class CNNTextClassifier(nn.Module):
    """1-D CNN with multi-scale filters (TextCNN) – fast, effective for classification."""
    def __init__(self, embedding_matrix, num_filters=100, filter_sizes=(3,4,5)):
        super().__init__()
        vocab_size, embedding_dim = embedding_matrix.shape
        self.embedding = nn.Embedding.from_pretrained(
            torch.FloatTensor(embedding_matrix), freeze=False)
        self.convs = nn.ModuleList([
            nn.Conv1d(embedding_dim, num_filters, fs) for fs in filter_sizes])
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(num_filters * len(filter_sizes), 1)

    def forward(self, x):
        x = self.embedding(x).permute(0, 2, 1)
        pooled = [torch.max(torch.relu(conv(x)), dim=2)[0] for conv in self.convs]
        x = self.dropout(torch.cat(pooled, dim=1))
        return torch.sigmoid(self.fc(x))


print('Model classes defined: RNNTextClassifierV2, LSTMTextClassifier, CNNTextClassifier')


## 3.5 Training Loop with Early Stopping


In [ ]:
def train_model(model, train_loader, val_loader, epochs=15, lr=1e-3, patience=3):
    model.to(device)
    optimizer  = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion  = nn.BCELoss()
    scheduler  = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)
    best_val_loss, best_state, no_improve = float('inf'), None, 0
    history = {'train_loss':[], 'val_loss':[], 'val_acc':[]}

    for epoch in range(epochs):
        model.train()
        train_losses = []
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            out  = model(X_batch).squeeze()
            loss = criterion(out, y_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses, preds, trues = [], [], []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                out = model(X_batch).squeeze()
                val_losses.append(criterion(out, y_batch).item())
                preds.extend((out > 0.5).cpu().numpy())
                trues.extend(y_batch.cpu().numpy())

        t_loss = np.mean(train_losses)
        v_loss = np.mean(val_losses)
        v_acc  = accuracy_score(trues, preds)
        history['train_loss'].append(t_loss)
        history['val_loss'].append(v_loss)
        history['val_acc'].append(v_acc)
        scheduler.step(v_loss)
        print(f'Epoch {epoch+1:02d}/{epochs} | '
              f'Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f} | Val Acc: {v_acc:.4f}')
        if v_loss < best_val_loss:
            best_val_loss = v_loss
            best_state    = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve    = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break

    model.load_state_dict(best_state)
    return model, history


def evaluate_dl(model, loader):
    model.eval()
    probs, preds, trues = [], [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            out = model(X_batch.to(device)).squeeze().cpu()
            probs.extend(out.numpy())
            preds.extend((out > 0.5).numpy().astype(int))
            trues.extend(y_batch.numpy())
    return np.array(trues), np.array(preds), np.array(probs)


def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history['train_loss'], label='Train', marker='o')
    axes[0].plot(history['val_loss'],   label='Val',   marker='s')
    axes[0].set_title(f'{title} – Loss'); axes[0].legend()
    axes[1].plot(history['val_acc'], label='Val Accuracy', marker='^', color='green')
    axes[1].set_title(f'{title} – Val Accuracy'); axes[1].legend()
    plt.tight_layout(); plt.show()

print('Training utilities ready.')


## 3.6 RNN / Bidirectional GRU with Word2Vec

The GRU reads token sequences in both directions and uses global average pooling
to produce a fixed-size representation for classification.


In [ ]:
print('=' * 55)
print('Training Bidirectional GRU + Word2Vec Embeddings')
print('=' * 55)
model_rnn = RNNTextClassifierV2(embedding_matrix_w2v)
model_rnn, history_rnn = train_model(
    model_rnn, train_loader, val_loader, epochs=15, lr=1e-3, patience=3)
plot_history(history_rnn, 'RNN/GRU (Word2Vec)')


In [ ]:
true_rnn, pred_rnn, prob_rnn = evaluate_dl(model_rnn, test_loader)
print('RNN Test Report:')
print(classification_report(true_rnn, pred_rnn, target_names=['Human','AI']))
print(f'ROC-AUC: {roc_auc_score(true_rnn, prob_rnn):.4f}')


## 3.7 LSTM with GloVe Embeddings

The LSTM uses its gating mechanism to remember or forget information over long sequences.
GloVe provides rich semantic initialisation from a large external corpus.


In [ ]:
print('=' * 50)
print('Training LSTM + GloVe Embeddings')
print('=' * 50)
model_lstm = LSTMTextClassifier(embedding_matrix_glove)
model_lstm, history_lstm = train_model(
    model_lstm, train_loader, val_loader, epochs=15, lr=1e-3, patience=3)
plot_history(history_lstm, 'LSTM (GloVe)')


In [ ]:
true_lstm, pred_lstm, prob_lstm = evaluate_dl(model_lstm, test_loader)
print('LSTM Test Report:')
print(classification_report(true_lstm, pred_lstm, target_names=['Human','AI']))
print(f'ROC-AUC: {roc_auc_score(true_lstm, prob_lstm):.4f}')


## 3.8 CNN (TextCNN) with Word2Vec Embeddings

Parallel convolutional filters of sizes 3, 4, 5 extract local n-gram features.
Global max pooling picks the strongest signal per filter across the sequence.


In [ ]:
print('=' * 50)
print('Training TextCNN + Word2Vec Embeddings')
print('=' * 50)
model_cnn = CNNTextClassifier(embedding_matrix_w2v)
model_cnn, history_cnn = train_model(
    model_cnn, train_loader, val_loader, epochs=15, lr=1e-3, patience=3)
plot_history(history_cnn, 'CNN (Word2Vec)')


In [ ]:
true_cnn, pred_cnn, prob_cnn = evaluate_dl(model_cnn, test_loader)
print('CNN Test Report:')
print(classification_report(true_cnn, pred_cnn, target_names=['Human','AI']))
print(f'ROC-AUC: {roc_auc_score(true_cnn, prob_cnn):.4f}')


---
# Section 4 – Evaluation & Comparison


## 4.1 Comprehensive Metrics Table (Test Set)


In [ ]:
y_test_np = y_test.values
results = {}

# Classical ML results
for name, pipeline, X_col in [
        ('SVM',           pipeline_svm, 'lemmatized'),
        ('Decision Tree', pipeline_dt,  'lemmatized'),
        ('AdaBoost',      pipeline_ada, 'lemmatized'),
    ]:
    yp  = pipeline.predict(X_test[X_col])
    ypr = pipeline.predict_proba(X_test[X_col])[:,1]
    p,r,f,_ = precision_recall_fscore_support(y_test_np, yp, average='weighted', zero_division=0)
    results[name] = dict(Type='Classical ML',
        Accuracy=accuracy_score(y_test_np, yp),
        Precision=p, Recall=r, F1=f,
        ROCAUC=roc_auc_score(y_test_np, ypr),
        y_pred=yp, y_prob=ypr, y_true=y_test_np)

# Deep Learning results
for name, (yt, yp, ypr) in [
        ('RNN/GRU (W2V)',  (true_rnn,  pred_rnn,  prob_rnn)),
        ('LSTM (GloVe)',   (true_lstm, pred_lstm, prob_lstm)),
        ('CNN (W2V)',      (true_cnn,  pred_cnn,  prob_cnn)),
    ]:
    p,r,f,_ = precision_recall_fscore_support(yt, yp, average='weighted', zero_division=0)
    results[name] = dict(Type='Deep Learning',
        Accuracy=accuracy_score(yt, yp),
        Precision=p, Recall=r, F1=f,
        ROCAUC=roc_auc_score(yt, ypr),
        y_pred=yp, y_prob=ypr, y_true=yt)

cmp_df = pd.DataFrame({
    k: {m: v for m,v in v.items() if m not in ['y_pred','y_prob','y_true']}
    for k,v in results.items()
}).T
cmp_df[['Accuracy','Precision','Recall','F1','ROCAUC']] = \
    cmp_df[['Accuracy','Precision','Recall','F1','ROCAUC']].astype(float).round(4)
cmp_df = cmp_df.sort_values('Accuracy', ascending=False)

print('=== All 6 Models – Test Set Performance ===')
display(cmp_df[['Type','Accuracy','Precision','Recall','F1','ROCAUC']])


In [ ]:
metrics_to_plot = ['Accuracy', 'F1', 'ROCAUC']
x = np.arange(len(results))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 6))
for i, metric in enumerate(metrics_to_plot):
    vals = [results[m][metric] for m in results]
    ax.bar(x + i*width, vals, width, label=metric)

ax.set_xticks(x + width)
ax.set_xticklabels(list(results.keys()), rotation=20, ha='right')
ax.set_ylim(0, 1.1)
ax.set_title('Model Comparison – Accuracy, F1, and ROC-AUC (Test Set)',
             fontsize=13, fontweight='bold')
ax.legend()
ax.axhline(0.5, color='red', linestyle='--', alpha=0.4, label='Random baseline')
plt.tight_layout()
plt.savefig('plots/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## 4.2 Confusion Matrices


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, (name, info) in enumerate(results.items()):
    cm   = confusion_matrix(info['y_true'], info['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['Human','AI'])
    disp.plot(ax=axes[i], colorbar=False, cmap='Blues')
    axes[i].set_title(name, fontsize=12, fontweight='bold')

plt.suptitle('Confusion Matrices – All 6 Models (Test Set)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('plots/all_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Save individual confusion matrix plots (filenames expected by app.py)
cf_name_map = {
    'SVM':          'SVM cf',
    'Decision Tree':'DT cf',
    'AdaBoost':     'AdaBoost CF',
    'RNN/GRU (W2V)':'RNN- w2v conf',
    'LSTM (GloVe)': 'LSTM Cf',
    'CNN (W2V)':    'CNN- w2v conf',
}
for name, fname in cf_name_map.items():
    fig, ax = plt.subplots(figsize=(5, 4))
    cm = confusion_matrix(results[name]['y_true'], results[name]['y_pred'])
    ConfusionMatrixDisplay(cm, display_labels=['Human','AI']).plot(
        ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(fname)
    plt.tight_layout()
    plt.savefig(f'plots/{fname}.png', dpi=150, bbox_inches='tight')
    plt.close()
print('Individual confusion matrix plots saved.')


## 4.3 ROC Curves


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ml_names = ['SVM', 'Decision Tree', 'AdaBoost']
dl_names = ['RNN/GRU (W2V)', 'LSTM (GloVe)', 'CNN (W2V)']

for name in ml_names:
    fpr, tpr, _ = roc_curve(results[name]['y_true'], results[name]['y_prob'])
    auc = results[name]['ROCAUC']
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
axes[0].plot([0,1],[0,1],'--',color='grey')
axes[0].set_title('ROC – Classical ML', fontsize=13, fontweight='bold')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].legend()

for name in dl_names:
    fpr, tpr, _ = roc_curve(results[name]['y_true'], results[name]['y_prob'])
    auc = results[name]['ROCAUC']
    axes[1].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
axes[1].plot([0,1],[0,1],'--',color='grey')
axes[1].set_title('ROC – Deep Learning', fontsize=13, fontweight='bold')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR'); axes[1].legend()

plt.suptitle('ROC Curves – All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/ML Modles ROC.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Individual ROC plots for app.py
roc_name_map = {
    'RNN/GRU (W2V)': 'RNN -w2v',
    'LSTM (GloVe)':  'LSTM Roc',
    'CNN (W2V)':     'CNN- W2v',
}
for name, fname in roc_name_map.items():
    fig, ax = plt.subplots(figsize=(6, 5))
    fpr, tpr, _ = roc_curve(results[name]['y_true'], results[name]['y_prob'])
    ax.plot(fpr, tpr, label=f"AUC={results[name]['ROCAUC']:.3f}")
    ax.plot([0,1],[0,1],'--',color='grey')
    ax.set_title(f'ROC – {name}'); ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.legend()
    plt.tight_layout()
    plt.savefig(f'plots/{fname}.png', dpi=150, bbox_inches='tight')
    plt.close()
print('Individual ROC plots saved.')


## 4.4 Feature Importance Analysis


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Decision Tree top 20 features
vec_dt  = pipeline_dt.named_steps['vectorizer']
tree_dt = pipeline_dt.named_steps['model']
imps = tree_dt.feature_importances_
feat = vec_dt.get_feature_names_out()
top  = np.argsort(imps)[-20:][::-1]
sns.barplot(x=imps[top], y=feat[top], ax=axes[0], palette='Blues_r')
axes[0].set_title('Decision Tree – Top 20 Features', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Gini Importance')

# SVM coefficients
svm_model = pipeline_svm.named_steps['model']
coefs = svm_model.coef_.toarray().flatten()
feat_svm = pipeline_svm.named_steps['vectorizer'].get_feature_names_out()
ai_top     = feat_svm[np.argsort(coefs)[-15:]]
human_top  = feat_svm[np.argsort(coefs)[:15]]
ai_coefs   = coefs[np.argsort(coefs)[-15:]]
hum_coefs  = coefs[np.argsort(coefs)[:15]]
all_feat   = np.concatenate([human_top, ai_top])
all_coefs  = np.concatenate([hum_coefs, ai_coefs])
colors = ['steelblue' if c < 0 else 'salmon' for c in all_coefs]
axes[1].barh(all_feat, all_coefs, color=colors)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('SVM Coefficients (Blue=Human indicator, Red=AI indicator)',
                  fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('plots/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Print top AI / Human words from SVM
top_ai_words    = feat_svm[np.argsort(coefs)[-15:][::-1]]
top_human_words = feat_svm[np.argsort(coefs)[:15]]
print('Top AI-indicator words (SVM):  ', ', '.join(top_ai_words))
print('Top Human-indicator words (SVM):', ', '.join(top_human_words))


## 4.5 Written Analysis

### Feature Comparison: TF-IDF vs Word Embeddings

**TF-IDF** creates a sparse, high-dimensional bag-of-words representation. It works
extremely well for this task because AI-generated text uses a consistently distinct vocabulary
(formal connectors like 'furthermore', 'moreover', precise hedging language) that TF-IDF
captures directly through term frequency differences.

**Word2Vec** (trained on our corpus) learns dense 100-dimensional embeddings where semantically
similar words cluster together. The GRU/CNN models using Word2Vec can generalise beyond exact
word matches, recognising stylistic patterns even when specific words differ.

**GloVe** (pre-trained on Wikipedia) provides rich semantic representations from a much larger
corpus. This helps the LSTM handle vocabulary not well-represented in our training data.

### ML vs Deep Learning Trade-offs

| Model | Approach | Key Advantage | Key Limitation |
|-------|----------|---------------|----------------|
| SVM | TF-IDF + linear kernel | High accuracy, LIME-explainable | No context window |
| Decision Tree | TF-IDF + splits | Fully interpretable, fast | Tends to overfit on sparse features |
| AdaBoost | TF-IDF + ensemble | Handles class noise well | Slow on large feature spaces |
| RNN/GRU | W2V + sequential | Captures word order context | Longer training, less interpretable |
| LSTM | GloVe + long-range | Handles long essays, pre-trained semantic knowledge | Slower than GRU |
| CNN | W2V + n-gram filters | Very fast, captures local phrases | Fixed filter size limits context |

### Key Insight

> SVM with TF-IDF achieves near-best performance on this task because AI-generated text is
> stylistically consistent — it favours formal connectors, hedging language, and structured
> vocabulary that TF-IDF captures as distinctive term-frequency patterns.
> Deep learning models are competitive but require significantly more compute for training.


---
## 5 – Save All Model Files

Run this cell after training. Models are saved to `models/` (same folder as `app.py`).


In [ ]:
# ─── Classical ML pipelines ───────────────────────────────────────────────
joblib.dump(pipeline_svm, 'models/svm_pipeline.pkl')
joblib.dump(pipeline_dt,  'models/decision_tree_pipeline.pkl')
joblib.dump(pipeline_ada, 'models/adaboost_pipeline.pkl')
print('ML pipelines saved.')

# ─── Keras tokenizer ──────────────────────────────────────────────────────
with open('models/tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
print('Tokenizer saved.')

# ─── Embedding matrices ───────────────────────────────────────────────────
np.save('models/embedding_matrix_w2v.npy',   embedding_matrix_w2v)
np.save('models/embedding_matrix_glove.npy', embedding_matrix_glove)
print('Embedding matrices saved.')

# ─── PyTorch DL weights ───────────────────────────────────────────────────
torch.save(model_rnn.state_dict(),  'models/RNN_w2v.pt')
torch.save(model_lstm.state_dict(), 'models/LSTM_glove.pt')
torch.save(model_cnn.state_dict(),  'models/CNN_w2v.pt')
print('DL model weights saved.')

print('\nAll files in models/:')
for f in sorted(os.listdir('models')):
    kb = os.path.getsize(f'models/{f}') / 1024
    print(f'  {f:<45} {kb:>8,.1f} KB')


---
## 6 – How to Run the Streamlit App

```bash
# 1. Navigate to the project folder (where app.py lives)
cd path/to/Muhammad_Haseeb_project_2

# 2. Install dependencies
pip install -r requirements.txt

# 3. Launch the app
streamlit run app.py
```

**Important:** If the `MAX_LEN` printed in Section 2.5 differs from 192,
open `app.py` and update the default value in `make_prediction()`:

```python
# Line ~178 in app.py
def make_prediction(text, model_choice, models, max_len_pad=YOUR_MAX_LEN):
```
